# Uncertainty Analysis: Field of View (FOV)
This notebook analyzes the impact of different Field of View (FOV) settings on slope estimation accuracy.
It includes:
1. Data preparation (panoramas)
2. Image transformation (generating perspective views with varying FOVs)
3. Result analysis and visualization

In [ ]:
import os
import shutil
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zensvi
from zensvi.transform import ImageTransformer

# Plotting settings
plt.rcParams.update({'font.size': 14})
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Configuration & Paths
BASE_DIR = 'uncertainty_analysis'

# Input Data
VALIDATION_SAMPLES_PATH = os.path.join(BASE_DIR, 'subsamples_result.gpkg')


# Working Directories
WORK_PANOS_DIR = os.path.join(BASE_DIR, 'panoramas')
OUTPUT_BASE_DIR = os.path.join(BASE_DIR, 'fov-panos')
RESULTS_DIR = os.path.join(BASE_DIR, 'fov-test')

# Output
FIGURE_SAVE_PATH = os.path.join(BASE_DIR, 'FOV_uncertainty.png')

# Parameters
FOVs = [60, 70, 80, 90, 100, 110, 120, 130, 140, 150]

In [ ]:
# Load validation samples
subsamples = gpd.read_file(VALIDATION_SAMPLES_PATH)
display(subsamples.head())

In [ ]:
def run_fov_transformation(input_dir, base_output_dir, fovs):
    """
    Transform panoramas to perspective images with varying FOVs.
    Also cleans up Direction_0 and Direction_180 images.
    """
    for fov in fovs:
        output_dir = os.path.join(base_output_dir, f'transformed_panoramas_{fov}FOV')
        print(f"Processing FOV: {fov}° -> {output_dir}")
        
        image_transformer = ImageTransformer(dir_input=input_dir, dir_output=output_dir)
        image_transformer.transform_images(
            style_list="perspective",
            FOV=fov,
            theta=90,
            phi=0,
            aspects=(10, 10),
            show_size=100
        )
        
        # Cleanup
        print(f"Cleaning up Direction_0 and Direction_180 for FOV {fov}...")
        os.system(f"find {output_dir} -name '*Direction_0*' -delete")
        os.system(f"find {output_dir} -name '*Direction_180*' -delete")

run_fov_transformation(WORK_PANOS_DIR, OUTPUT_BASE_DIR, FOVs)

In [ ]:
def search_csv_files(directory, keyword=None):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                if keyword is None or keyword in file:
                    csv_files.append(os.path.join(root, file))
    return csv_files

def load_fov_results(base_path, fovs, sample_df):
    """Load and merge results from different FOV runs."""
    merged_df = sample_df[['pano_id', 'slope_1m', 'slope_30m']].copy()
    merged_df = merged_df.set_index('pano_id')
    
    available_ratios = []
    
    for fov in fovs:
        search_path = os.path.join(base_path, f'chunk_{fov}FOV')
        fov_files = search_csv_files(search_path, keyword='adjust')
        
        if fov_files:
            # Assuming the first file is the one we want
            fov_df = pd.read_csv(fov_files[0])
            
            # Calculate availability ratio
            n = len(fov_df)
            # Assuming 2000 is the total count as per original code
            available_ratios.append(n / 2000 * 100) 
            
            fov_df.set_index('pano_id', inplace=True)
            fov_df = fov_df[['adjusted_angle']]
            fov_df = fov_df.rename(columns={'adjusted_angle': f'FOV{fov}_adjusted_angle'})
            
            # Drop duplicates
            fov_df = fov_df[~fov_df.index.duplicated(keep='first')]
            
            merged_df = merged_df.join(fov_df, how='left')
            
            # Calculate difference
            merged_df[f'difference_FOV{fov}'] = merged_df[f'FOV{fov}_adjusted_angle'] - merged_df['slope_1m']
        else:
            print(f"No results found for FOV {fov}")
            available_ratios.append(0)
            
    return merged_df, available_ratios

# Run analysis
analyzed_df, availability = load_fov_results(RESULTS_DIR, FOVs, subsamples)
display(analyzed_df.head())

In [ ]:
def plot_fov_uncertainty(sample_df, availability_pct, angles, save_path=None):
    import matplotlib.ticker as mticker

    # Filter valid angles
    plot_angles = [a for a in angles if f'difference_FOV{a}' in sample_df.columns]
    if not plot_angles:
        print("No data to plot.")
        return

    # Prepare data for plotting
    rename_map = {f'difference_FOV{a}': f"{a}°" for a in plot_angles}
    plot_df = (
        sample_df[[f'difference_FOV{a}' for a in plot_angles]]
        .rename(columns=rename_map)
        .melt(var_name='FOV angle (°)', value_name='Slope difference (°)')
    )
    
    order_labels = [f"{a}°" for a in plot_angles]
    
    # Plotting
    sns.set_theme(style='whitegrid', context='talk')
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[2, 1], hspace=0.32)
    
    # Boxplot
    ax_box = fig.add_subplot(gs[0])
    sns.boxplot(
        data=plot_df,
        x='FOV angle (°)',
        y='Slope difference (°)',
        ax=ax_box,
        width=0.35,
        fliersize=2.2,
        linewidth=1.2,
        medianprops={'color': '#0F3460', 'linewidth': 2},
        whiskerprops={'color': '#0F3460', 'linewidth': 1.2},
        capprops={'color': '#0F3460', 'linewidth': 1.2},
        showfliers=False,
        showcaps=False,
        palette=sns.cubehelix_palette(len(order_labels), start=0.35, rot=-0.15, light=0.85, dark=0.25)
    )
    
    ax_box.axhline(0, color='#5B6270', linestyle='--', linewidth=1, alpha=0.8, zorder=0)
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Slope difference (°)', labelpad=12)
    ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
    
    # Availability Line Plot
    ax_line = fig.add_subplot(gs[1], sharex=ax_box)
    positions = np.arange(len(order_labels))
    
    ax_line.plot(positions, availability_pct, color=sns.color_palette('crest', n_colors=7)[-2], marker='o', linewidth=2.4, label='Availability')
    ax_line.set_xlabel('FOV angle (°)', labelpad=12)
    ax_line.set_ylabel('Available imagery (%)', labelpad=12)
    ax_line.set_xticks(positions)
    ax_line.set_xticklabels(order_labels)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot
plot_fov_uncertainty(
    analyzed_df,
    availability,
    FOVs,
    save_path=FIGURE_SAVE_PATH
)